In [ ]:
import pandas as pd
import numpy as np


In [ ]:
customers = pd.read_csv('olist_customers_dataset[1].csv')
orders = pd.read_csv('olist_orders_dataset[1].csv')
items = pd.read_csv('olist_order_items_dataset[1].csv')

In [ ]:
df = pd.merge(orders, customers, on='customer_id')
df = pd.merge(df, items, on='order_id')

In [ ]:
df = df[['customer_unique_id', 'order_purchase_timestamp', 'price']]
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

In [ ]:
df.head()

,customer_unique_id,order_purchase_timestamp,price
0,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,29.99
1,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,118.70
2,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,159.90
3,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,45.00
4,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,19.90


In [ ]:
!pip install lifetimes
from lifetimes import GammaGammaFitter,BetaGeoFitter
from lifetimes.utils import summary_data_from_transaction_data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 10.2 MB/s eta 0:00:00


In [ ]:
rfm = summary_data_from_transaction_data(
    df,
    'customer_unique_id',
    'order_purchase_timestamp',
    monetary_value_col='price',
    observation_period_end='2018-09-03'
)

In [ ]:
rfm = rfm[rfm['frequency'] > 0]

In [ ]:
bgf = BetaGeoFitter()
bgf.fit(rfm['frequency'], rfm['recency'], rfm['T'])

<lifetimes.BetaGeoFitter: fitted with 2085 subjects, a: 2.21, alpha: 2367.70, b: 0.26, r: 21.52>

In [ ]:
rfm['predicted_purchases_30d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    30, rfm['frequency'], rfm['recency'], rfm['T'])

In [ ]:
returning_customers = rfm[rfm['frequency'] > 0]
ggf = GammaGammaFitter()
ggf.fit(returning_customers['frequency'], returning_customers['monetary_value'])

<lifetimes.GammaGammaFitter: fitted with 2085 subjects, p: 2.81, q: 2.94, v: 88.94>

In [ ]:
rfm['expected_avg_value'] = ggf.conditional_expected_average_profit(
    rfm['frequency'],
    rfm['monetary_value']
)

In [ ]:
# Calculate the expected value for the next 12 months
rfm['clv_12m'] = ggf.customer_lifetime_value(
    bgf,
    rfm['frequency'],
    rfm['recency'],
    rfm['T'],
    rfm['monetary_value'],
    time=12,
    discount_rate=0.01
)

In [ ]:
rfm.head()

,frequency,recency,T,monetary_value,predicted_purchases_30d,expected_avg_value,clv_12m
customer_unique_id,,,,,,,
004288347e5e88a27ded2bb23747066c,1.0,171.0,403.0,87.90,0.003651,104.732760,2.221086
004b45ec5c64187465168251cd1c9c2f,1.0,267.0,367.0,27.00,0.011069,68.664798,4.386226
0058f300f57d7b93c477a131a59b36c3,1.0,31.0,196.0,72.58,0.006221,95.659506,3.323907
00a39521eb40f7012db50455bf083460,1.0,11.0,103.0,11.55,0.011971,59.514551,3.903636
011575986092c30523ecb71ff10cb473,1.0,60.0,198.0,63.90,0.007948,90.518785,4.020096


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = ['frequency', 'recency', 'monetary_value', 'clv_12m']
data_for_clustering = rfm[features]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(data_for_clustering)

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['segment_id'] = kmeans.fit_predict(scaled_features)

In [ ]:
rfm.to_csv('olist_customer_segmentation.csv', index=True)
print("Analysis has been exported to 'olist_customer_segmentation.csv'!")

Analysis has been exported to 'olist_customer_segmentation.csv'!


In [ ]:
New_Data=pd.read_csv('olist_customer_segmentation.csv')
New_Data.head()

,customer_unique_id,frequency,recency,T,monetary_value,predicted_purchases_30d,expected_avg_value,clv_12m,segment_id
0,004288347e5e88a27ded2bb23747066c,1.0,171.0,403.0,87.90,0.003651,104.732760,2.221086,3
1,004b45ec5c64187465168251cd1c9c2f,1.0,267.0,367.0,27.00,0.011069,68.664798,4.386226,2
2,0058f300f57d7b93c477a131a59b36c3,1.0,31.0,196.0,72.58,0.006221,95.659506,3.323907,3
3,00a39521eb40f7012db50455bf083460,1.0,11.0,103.0,11.55,0.011971,59.514551,3.903636,3
4,011575986092c30523ecb71ff10cb473,1.0,60.0,198.0,63.90,0.007948,90.518785,4.020096,3


In [ ]:
# Load your segmentation results
df = pd.read_csv('olist_customer_segmentation.csv')

# Define the mapping based on your K-Means results
# Note: You may need to verify which ID corresponds to which group by looking at your averages
segment_map = {
    0: "Champions",
    1: "At-Risk",
    2: "Potential Loyalists",
    3: "Hibernating"
}


df['segment_name'] = df['segment_id'].map(segment_map)


df = df.rename(columns={
    'T': 'customer_age_days',
    'clv_12m': 'predicted_clv_1yr',
    'predicted_purchases_30d': 'next_month_purchase_prob'
})

df.to_csv('polished_customer_segments.csv', index=False)